# Control de ejecucion guppy: simulador local y Nexus/Selene

Cuaderno con dos modos, controlados por `ALLOW_NEW_EXECUTION`:

- **RUN NUEVO** (`True`): ejecuta una simulacion nueva (local o remota en Nexus/Selene).
- **CONSULTA** (`False`): lista y carga los resultados de un job ya terminado.

La logica de control de cada celda vive en `funciones_nexus.py`; aqui solo quedan los parametros a modificar y las llamadas.

## Configuracion y conexion a Nexus

In [ ]:
import uuid
import pandas as pd
import qnexus as qnx
from IPython.display import display
from funciones_nexus import *

PROJECT_NAME = "guppy-kernel-encoding"          # Nombre del proyecto

# Interruptor maestro del cuaderno:
#   True  -> RUN NUEVO: se ejecuta una simulacion nueva (local o remota).
#            NO se consultan jobs existentes (las celdas de listado/seleccion no aplican).
#   False -> CONSULTA: se lista y se carga un job ya terminado especifico.
#            NO se ejecuta nada nuevo (las celdas de compilacion/ejecucion no aplican).
ALLOW_NEW_EXECUTION = True

# Destino del run nuevo (solo se usa cuando ALLOW_NEW_EXECUTION = True):
#   "local"        -> simulador local (no toca Nexus)
#   "nexus_selene" -> job remoto en Nexus/Selene
EXECUTION_TARGET = "nexus_selene"

n_shots = 100       # Numero de iteraciones a realizar

In [ ]:
suffix = uuid.uuid4().hex[:8]
# Estado de la ejecucion cargada/activa; se rellena en las celdas siguientes
# segun el modo (RUN NUEVO local/remoto o CONSULTA de un job existente).
(RESULT_SOURCE, local_result, local_counts, local_run_id, sim_job_ref, sim_result, sim_counts, sim_result_ids, selected_counts, selected_job_ref, selected_job_name, selected_result_ids,) = (None,) * 12

project = conectar_nexus(PROJECT_NAME)

modo = "RUN NUEVO" if ALLOW_NEW_EXECUTION else "CONSULTA de job existente"
destino = f"\nDestino: {EXECUTION_TARGET}" if ALLOW_NEW_EXECUTION else ""
print(f"Conexion con Nexus comprobada.\nProyecto: {PROJECT_NAME}\nProject ID: {project.id}\nModo: {modo}{destino}")

## Modo CONSULTA
### listado de jobs existentes
Solo aplica si `ALLOW_NEW_EXECUTION = False`.

In [ ]:
execution_job_refs, job_selector = listar_jobs_si_corresponde(ALLOW_NEW_EXECUTION, project, PROJECT_NAME)
if job_selector is not None:
    display(job_selector)

### carga de resultados del job seleccionado

In [ ]:
(
    selected_job_ref, selected_job_name, selected_counts, selected_result_ids,
    sim_result, sim_counts, result_source, selected_results_df,
) = cargar_job_seleccionado(ALLOW_NEW_EXECUTION, job_selector, execution_job_refs)

if result_source is not None:
    RESULT_SOURCE = result_source
    display(selected_results_df)

## Definicion del circuito guppy

In [ ]:
from guppylang import guppy
from guppylang.std.builtins import result
from guppylang.std.quantum import cx, h, measure, qubit

@guppy
def encode_logical_plus() -> None:
    data, parity = qubit(), qubit()
    h(data)
    cx(data, parity)
    result("logical[0]", measure(data))
    result("logical[1]", measure(parity))

encode_logical_plus.check()

## Modo RUN NUEVO
### compilacion y subida del HUGR
Solo aplica para destino `nexus_selene`.

In [ ]:
hugr_binary, ref_hugr = compilar_si_corresponde(
    ALLOW_NEW_EXECUTION, EXECUTION_TARGET, encode_logical_plus, suffix
)

### ejecucion (local o remota)

In [ ]:
local_result, local_counts, local_run_id, sim_job_ref, result_source = ejecutar_si_corresponde(
    ALLOW_NEW_EXECUTION, EXECUTION_TARGET, encode_logical_plus, ref_hugr, n_shots, suffix
)
if result_source is not None:
    RESULT_SOURCE = result_source

### Consulta de un job remoto en curso
Solo aplica si se envio un job remoto en la celda anterior. 
Se puede ejecutar para observar el status del job enviado. 

Reejecutar cuando ya se tenga el estatus completo para obtener la tabla de resultados

In [ ]:
actualizado, nuevo_sim_result, nuevo_sim_counts, nuevo_sim_result_ids, result_source = (
    consultar_job_remoto(sim_job_ref)
)
if actualizado:
    sim_result, sim_counts, sim_result_ids, RESULT_SOURCE = (
        nuevo_sim_result, nuevo_sim_counts, nuevo_sim_result_ids, result_source
    )

## Guardado de resultados en CSV

In [ ]:
# Guarda la ejecucion cargada (local, job seleccionado o job remoto nuevo) en un CSV (";")

ruta_run = guardar_resultado_actual(
    RESULT_SOURCE, n_shots=n_shots,
    local_counts=local_counts, local_run_id=local_run_id,
    selected_counts=selected_counts, selected_job_ref=selected_job_ref,
    selected_job_name=selected_job_name, selected_result_ids=selected_result_ids,
    sim_counts=sim_counts, sim_job_ref=sim_job_ref, sim_result_ids=sim_result_ids,
)

print(f"Ejecucion guardada en: {ruta_run}")
pd.read_csv(ruta_run, sep=";")